# 사례 2~4 — 박스 카운팅 · 운동/낙상 포즈 · 주차장과 대기열

**파일**: `yolo_day1_case_demos_v1.1.ipynb` · **환경**: Google Colab (T4 GPU)

세 사례 모두 **"YOLO 출력 + 간단한 규칙 = 현장에서 쓰는 기능"** 구조입니다. 포트폴리오 주제를 고를 때 참고하세요.

| 사례 | YOLO 태스크 | 학습 필요 | 붙이는 규칙 | 결과물 |
|---|---|---|---|---|
| 2. 물류 박스 카운팅 | 분할(segment) + 추적(track) | 예 (10 에포크) | 사진당 개수 · 선을 넘으면 +1 | 개수가 찍힌 사진, 집계 영상 |
| 3. 운동·낙상 | 포즈(pose) | 아니오 | 관절 각도 · 몸통 기울기와 낙하 속도 | 횟수 카운터, 낙상 경보 |
| 4. 주차장·대기열 | 탐지 + 추적 | 아니오 | 영역(ROI) 안의 대수 | 빈자리 수, 대기 인원, 요약 CSV |

> 영상은 Ultralytics가 공개한 2~3초짜리 예제입니다. 자기 영상으로 바꾸려면 `VIDEO_...` 경로만 바꾸면 됩니다.

## 0장. 설치와 예제 영상 내려받기

In [ ]:
!pip install -q ultralytics shapely lap

In [ ]:
from pathlib import Path
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO, solutions
from ultralytics.utils import ASSETS_URL
from ultralytics.utils.downloads import safe_download

for f in ["solutions_ci_demo.mp4", "solution_ci_pose_demo.mp4", "solution_ci_parking_demo.mp4",
          "solution_ci_parking_areas.json", "solutions_ci_parking_model.pt"]:
    if not Path(f).exists():
        safe_download(url=f"{ASSETS_URL}/{f}", dir=".")

VIDEO_CROWD, VIDEO_PUSHUP, VIDEO_PARKING = "solutions_ci_demo.mp4", "solution_ci_pose_demo.mp4", "solution_ci_parking_demo.mp4"
OUT = Path("demo_outputs"); OUT.mkdir(exist_ok=True)

def video_info(path):
    cap = cv2.VideoCapture(path)
    return int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS), int(cap.get(3)), int(cap.get(4))

for v in (VIDEO_CROWD, VIDEO_PUSHUP, VIDEO_PARKING):
    n, fps, w, h = video_info(v); print(f"{v}: {n} frames · {fps:.0f} fps · {w}x{h}")

In [ ]:
# 영상에 솔루션을 돌려 결과 영상을 저장하고, 프레임마다 결과를 기록하는 공통 함수
def run_solution(sol, video, out_name, max_frames=None):
    cap = cv2.VideoCapture(video)
    n, fps, w, h = video_info(video)
    writer = cv2.VideoWriter(str(OUT / out_name), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    logs, frames, k = [], [], 0
    while True:
        ok, im = cap.read()
        if not ok or (max_frames and k >= max_frames): break
        r = sol(im)
        writer.write(r.plot_im); logs.append(r); k += 1
        if k in (1, n // 2, n - 1): frames.append(r.plot_im.copy())
    writer.release()
    return logs, frames

def show_frames(frames, titles=None):
    fig, axes = plt.subplots(1, len(frames), figsize=(6 * len(frames), 4))
    for i, (ax, f) in enumerate(zip(np.atleast_1d(axes), frames)):
        ax.imshow(f[..., ::-1]); ax.axis("off")
        if titles: ax.set_title(titles[i])
    plt.show()

---
## 사례 2. 물류센터 박스 카운팅

**문제**: 입고된 팔레트 사진 한 장에 박스가 몇 개인지, 컨베이어로 몇 개가 지나갔는지 자동으로 센다.

**왜 분할(segment)인가**: 박스는 서로 딱 붙어 쌓입니다. 사각형 박스만으로는 경계가 겹쳐 하나로 합쳐지기 쉽고, 마스크(픽셀 단위 윤곽)는 붙은 박스를 더 잘 갈라 줍니다.

### 2-1. 학습 — Ultralytics Package-seg (학습 1,920 · 검증 188 · 테스트 89장, 클래스 `package`)

In [ ]:
EPOCHS_PKG = 10     # T4 기준 약 5~8분. CPU라면 1
IMGSZ_PKG = 640     # CPU라면 320

seg = YOLO("yolo26n-seg.pt")
res = seg.train(data="package-seg.yaml", epochs=EPOCHS_PKG, imgsz=IMGSZ_PKG, batch=16, seed=0,
                project="runs_pkg", name="seg", exist_ok=True)
PKG_RUN = Path(res.save_dir)
pkg = YOLO(PKG_RUN / "weights" / "best.pt")
mp = pkg.val(data="package-seg.yaml", imgsz=IMGSZ_PKG, verbose=False)
print(f"박스(bbox) mAP50 {mp.box.map50:.3f} · 마스크(mask) mAP50 {mp.seg.map50:.3f}")

### 2-2. 사진 한 장의 박스 수 — 정답 개수와 비교

In [ ]:
from ultralytics.data.utils import check_det_dataset
pdata = check_det_dataset("package-seg.yaml")
test_dir = Path(pdata["test"]); lbl_dir = Path(str(test_dir).replace("images", "labels"))
test_imgs = sorted(test_dir.glob("*"))

rows = []
for img, r in zip(test_imgs, pkg.predict(test_imgs, conf=0.25, imgsz=IMGSZ_PKG, verbose=False)):
    gt = len([l for l in (lbl_dir / f"{img.stem}.txt").read_text().splitlines() if l.strip()])
    rows.append({"image": img.name, "true_count": gt, "pred_count": len(r.boxes)})
cnt = pd.DataFrame(rows); cnt["error"] = cnt.pred_count - cnt.true_count
cnt.to_csv(OUT / "package_counts.csv", index=False)
print(f"테스트 {len(cnt)}장 · 개수가 정확히 맞은 사진 {int((cnt.error == 0).sum())}장 ({(cnt.error == 0).mean():.0%}) · 평균 절대 오차 {cnt.error.abs().mean():.2f}개")
cnt.sort_values("error", key=abs, ascending=False).head(5)

In [ ]:
# 개수가 가장 많이 틀린 사진과 잘 맞은 사진
worst = cnt.reindex(cnt.error.abs().sort_values(ascending=False).index).iloc[0]
good = cnt[cnt.error == 0].sort_values("true_count", ascending=False).iloc[0] if (cnt.error == 0).any() else cnt.iloc[0]
picks = [(good, "count OK"), (worst, "count wrong")]
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (row, tag) in zip(axes, picks):
    r = pkg.predict(test_dir / row.image, conf=0.25, imgsz=IMGSZ_PKG, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
    ax.set_title(f"{tag}: true {row.true_count} / pred {row.pred_count}")
fig.savefig(OUT / "package_count_examples.jpg", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# 틀린 사진의 '정답 라벨'을 직접 그려 봅니다 (초록 = 정답 윤곽)
row = worst
im = cv2.imread(str(test_dir / row.image)); H, W = im.shape[:2]
for line in (lbl_dir / f"{Path(row.image).stem}.txt").read_text().splitlines():
    pts = (np.array(list(map(float, line.split()[1:]))).reshape(-1, 2) * [W, H]).astype(int)
    cv2.polylines(im, [pts], True, (0, 255, 0), 3)
plt.figure(figsize=(7, 7)); plt.imshow(im[..., ::-1]); plt.axis("off"); plt.title(f"labels only: {row.true_count} packages"); plt.show()

**읽어 볼 점**: 정답 라벨은 **컨베이어 위의 박스에만** 붙어 있고, 사람이 들고 있는 박스나 카트 안 박스에는 없습니다. 모델은 박스처럼 보이는 것을 다 찾았으니, "틀렸다"기보다 **무엇을 셀지 정하지 않은 것**이 문제입니다. 현장이라면 "컨베이어 영역 안의 박스만 센다"는 **영역(ROI) 규칙**을 붙입니다 — 사례 4에서 다시 나옵니다.

In [ ]:
# 확신도 기준(conf)을 바꾸면 개수 오차가 어떻게 달라질까
sweep = []
for conf in (0.25, 0.4, 0.5, 0.6):
    pred = [len(r.boxes) for r in pkg.predict(test_imgs, conf=conf, imgsz=IMGSZ_PKG, verbose=False)]
    e = np.array(pred) - cnt.true_count.values
    sweep.append({"conf": conf, "exact_ratio": round(float(np.mean(e == 0)), 2), "mae": round(float(np.abs(e).mean()), 2), "mean_signed_error": round(float(e.mean()), 2)})
sweep = pd.DataFrame(sweep); sweep.to_csv(OUT / "package_conf_sweep.csv", index=False)
sweep

**질문**: 어느 기준에서 오차가 가장 작나요? 평균 오차가 +(많이 셈)인지 −(적게 셈)인지도 보세요. 물류 현장에서 "박스를 덜 세는 것"과 "더 세는 것" 중 어느 쪽이 더 곤란할까요? — 이 기준값을 왜 골랐는지가 포트폴리오의 설계 판단이 됩니다.

### 2-3. 컨베이어처럼 '선을 넘으면 +1' — 추적(track)으로 집계

사진마다 개수를 세면 같은 박스를 여러 번 세게 됩니다. 영상에서는 **추적**으로 물체마다 ID를 붙이고, 그 ID가 선을 넘을 때 한 번만 셉니다. 공개된 박스 영상이 없어 사람이 걷는 영상으로 원리를 보여 줍니다. 박스 영상에서는 `model=`을 위에서 학습한 `pkg` 모델로, `classes=[0]`(package)으로 바꾸면 같은 코드가 그대로 동작합니다.

In [ ]:
n, fps, w, h = video_info(VIDEO_CROWD)
LINE = [(0, h // 2), (w, h // 2)]                       # 화면 가운데 가로선
counter = solutions.ObjectCounter(model="yolo26n.pt", region=LINE, classes=[0], show=False, verbose=False)
logs, frames = run_solution(counter, VIDEO_CROWD, "line_counting.mp4")
last = logs[-1]
print(f"선을 넘은 수 — IN {last.in_count} · OUT {last.out_count} · 추적한 사람 {last.total_tracks}명 ({n}프레임)")
show_frames(frames, ["start", "middle", "end"])

**포트폴리오 포인트**: "프레임마다 세면 중복 → 추적 ID로 선 통과 시 1회만 집계"라는 설계 판단 한 줄이 이 사례의 핵심입니다. 오차 표(`package_counts.csv`)와 함께 쓰면 "정확도 ○%, 오차 원인 ○○"까지 설명할 수 있습니다.

---
## 사례 3. 운동 자세 카운터와 낙상 감지 — 포즈(pose)

포즈 모델은 사람마다 **관절 점 17개**(코·눈·귀·어깨·팔꿈치·손목·엉덩이·무릎·발목)의 좌표를 냅니다. 학습 없이 사전학습 모델을 그대로 쓰고, 그 좌표에 **규칙**을 붙입니다.

### 3-1. 관절 점 보기

In [ ]:
pose = YOLO("yolo26n-pose.pt")
cap = cv2.VideoCapture(VIDEO_PUSHUP); ok, frame = cap.read()
r = pose.predict(frame, verbose=False)[0]
kp = r.keypoints.xy[0].cpu().numpy()
KP_NAMES = ["nose", "l_eye", "r_eye", "l_ear", "r_ear", "l_shoulder", "r_shoulder", "l_elbow", "r_elbow",
            "l_wrist", "r_wrist", "l_hip", "r_hip", "l_knee", "r_knee", "l_ankle", "r_ankle"]
print(pd.DataFrame(kp.round(0), index=KP_NAMES, columns=["x", "y"]).T)
Image.fromarray(r.plot()[..., ::-1])

### 3-2. 운동 횟수 세기 — 팔꿈치 각도 규칙 (Ultralytics AIGym)

어깨(6)–팔꿈치(8)–손목(10) 세 점이 이루는 각도가 **145° 위로 펴졌다가 100° 아래로 굽혀지면 1회**로 셉니다.

In [ ]:
gym = solutions.AIGym(model="yolo26n-pose.pt", kpts=[6, 8, 10], up_angle=145.0, down_angle=100.0, show=False, verbose=False)
logs, frames = run_solution(gym, VIDEO_PUSHUP, "pushup_counter.mp4")
angles = [l.workout_angle[0] if l.workout_angle else np.nan for l in logs]
print(f"팔굽혀펴기 횟수: {logs[-1].workout_count}")
plt.figure(figsize=(9, 3)); plt.plot(angles, color="#2E5E8C")
plt.axhline(145, ls="--", color="gray"); plt.axhline(100, ls="--", color="#E0602D")
plt.title("elbow angle per frame (up 145 / down 100)"); plt.xlabel("frame"); plt.ylabel("deg"); plt.show()
show_frames(frames, ["start", "middle", "end"])

### 3-3. 낙상 감지 — 규칙 하나로는 왜 부족한가

간단한 규칙부터 만들어 봅니다: **"몸통이 누워 있으면(어깨–엉덩이 선이 수평에 가까우면) 낙상"**. 이 규칙을 팔굽혀펴기 영상에 걸면 어떻게 될까요?

In [ ]:
def torso_metrics(kp):
    sh = (kp[5] + kp[6]) / 2; hip = (kp[11] + kp[12]) / 2
    angle = np.degrees(np.arctan2(abs(sh[1] - hip[1]), abs(sh[0] - hip[0]) + 1e-6))   # 0° = 수평(누움), 90° = 수직(섬)
    return angle, hip[1]

rows = []
cap = cv2.VideoCapture(VIDEO_PUSHUP); n, fps, w, h = video_info(VIDEO_PUSHUP); k = 0
while True:
    ok, im = cap.read()
    if not ok: break
    r = pose.predict(im, verbose=False)[0]
    if len(r.keypoints) and r.keypoints.conf is not None:
        kp = r.keypoints.xy[0].cpu().numpy(); ang, hip_y = torso_metrics(kp)
        rows.append({"frame": k, "torso_angle": ang, "hip_y": hip_y})
    k += 1
tor = pd.DataFrame(rows)

# 규칙 1: 몸통 각도 < 30° 이면 낙상
tor["rule1_fall"] = tor.torso_angle < 30
# 규칙 2: 규칙 1 + 0.5초 안에 엉덩이가 화면 높이의 15% 이상 '갑자기' 내려감
win = max(1, int(fps * 0.5))
tor["hip_drop"] = tor.hip_y - tor.hip_y.shift(win).bfill()
tor["rule2_fall"] = tor.rule1_fall & (tor.hip_drop > 0.15 * h)
tor[["torso_angle", "hip_drop"]] = tor[["torso_angle", "hip_drop"]].round(1)
tor.to_csv(OUT / "fall_rule_log.csv", index=False)
print(f"규칙 1(누워 있음)이 낙상이라고 한 프레임: {int(tor.rule1_fall.sum())}/{len(tor)}")
print(f"규칙 2(누워 있음 + 갑자기 떨어짐)이 낙상이라고 한 프레임: {int(tor.rule2_fall.sum())}/{len(tor)}")
tor.head()

**읽어 볼 점**: 팔굽혀펴기 자세는 몸통이 수평이라 **규칙 1은 거의 모든 프레임을 낙상으로 오판**합니다. 규칙 2처럼 **시간 축**("갑자기 떨어졌는가")을 더하면 오탐이 사라집니다. 실제 서비스에서는 여기에 "낙상 후 일정 시간 일어나지 않음"을 더하고, 실제 낙상 영상으로 기준값을 맞춥니다.

**포트폴리오 포인트**: "단순 규칙의 오탐을 직접 발견하고 시간 조건을 추가해 줄였다"는 경험은 모델 정확도 숫자보다 설득력 있는 이야기입니다.

---
## 사례 4. 주차장 빈자리와 매장 대기열 — 영역(ROI) 규칙

사전학습 모델에 **영역**만 그려 주면 바로 쓸 수 있는 사례입니다. 학습 없이도 결과가 나오기 때문에, 포트폴리오에서는 "어떤 영역을 어떻게 정했고 어디서 틀렸는지"가 핵심이 됩니다.

### 4-1. 주차장 빈자리 세기 (Ultralytics ParkingManagement)
주차면마다 꼭짓점 4개로 다각형을 정의한 JSON 파일과, 드론 시점(VisDrone) 데이터로 학습된 탐지 모델을 씁니다.

In [ ]:
import json
areas = json.load(open("solution_ci_parking_areas.json"))
print(f"주차면 {len(areas)}개 · 첫 주차면 꼭짓점: {areas[0]['points']}")
print("모델 클래스:", YOLO("solutions_ci_parking_model.pt").names)
# 주차면 좌표는 직접 찍을 수도 있습니다: solutions.ParkingPtsSelection() (로컬 PC의 GUI 도구)

In [ ]:
park = solutions.ParkingManagement(model="solutions_ci_parking_model.pt", json_file="solution_ci_parking_areas.json", show=False, verbose=False)
logs, frames = run_solution(park, VIDEO_PARKING, "parking.mp4")
pk = pd.DataFrame([{"frame": i, "filled": l.filled_slots, "available": l.available_slots} for i, l in enumerate(logs)])
print(f"첫 프레임 — 찬 자리 {pk.filled.iloc[0]} · 빈자리 {pk.available.iloc[0]}")
print(f"마지막 프레임 — 찬 자리 {pk.filled.iloc[-1]} · 빈자리 {pk.available.iloc[-1]} (총 {len(areas)}면)")
plt.figure(figsize=(9, 3)); plt.plot(pk.frame, pk.available, color="#E0602D", label="available"); plt.plot(pk.frame, pk.filled, color="#2E5E8C", label="filled")
plt.xlabel("frame"); plt.ylabel("slots"); plt.legend(); plt.title("parking slots over time"); plt.show()
show_frames(frames, ["start", "middle", "end"])

**질문**: COCO로 학습된 일반 모델(`yolo26n.pt`)을 쓰면 어떻게 될까요? 위에서 내려다본 차는 COCO 사진 속 차와 모양이 다릅니다. **카메라 시점에 맞는 데이터로 학습된 모델**이 필요한 이유입니다. 아래 셀로 직접 비교해 보세요.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PARKING); ok, im = cap.read()
coco_cars = sum(int(c) in (2, 5, 7) for c in YOLO("yolo26n.pt").predict(im, conf=0.25, verbose=False)[0].boxes.cls)   # car, bus, truck
drone_cars = sum(int(c) in (3, 4, 5) for c in YOLO("solutions_ci_parking_model.pt").predict(im, conf=0.25, verbose=False)[0].boxes.cls)  # car, van, truck
print(f"첫 프레임에서 찾은 차량 — COCO 일반 모델 {coco_cars}대 · 드론 시점 모델 {drone_cars}대")

### 4-2. 매장 대기열 인원 (Ultralytics QueueManager)
계산대 앞 영역을 다각형으로 정하고, 그 안에 있는 사람 수를 프레임마다 셉니다. 2-3과 같은 군중 영상을 씁니다.

In [ ]:
n, fps, w, h = video_info(VIDEO_CROWD)
QUEUE_AREA = [(int(w * 0.15), int(h * 0.3)), (int(w * 0.85), int(h * 0.3)), (int(w * 0.85), int(h * 0.85)), (int(w * 0.15), int(h * 0.85))]
queue = solutions.QueueManager(model="yolo26n.pt", region=QUEUE_AREA, classes=[0], show=False, verbose=False)
logs, frames = run_solution(queue, VIDEO_CROWD, "queue.mp4")
q = pd.DataFrame([{"frame": i, "queue_count": l.queue_count} for i, l in enumerate(logs)])
print(f"영역 안 인원 — 평균 {q.queue_count.mean():.1f}명 · 최대 {q.queue_count.max()}명")
show_frames(frames, ["start", "middle", "end"])

In [ ]:
# 대시보드에 올릴 요약 표 (열 이름은 영문 — Excel에서 깨지지 않습니다)
summary = pd.DataFrame([
    {"case": "parking", "metric": "available_slots_last", "value": int(pk.available.iloc[-1])},
    {"case": "parking", "metric": "filled_slots_last", "value": int(pk.filled.iloc[-1])},
    {"case": "queue", "metric": "queue_count_mean", "value": round(float(q.queue_count.mean()), 1)},
    {"case": "queue", "metric": "queue_count_max", "value": int(q.queue_count.max())},
])
summary.to_csv(OUT / "roi_summary.csv", index=False)
summary

---
## 결과물 모아 내려받기

In [ ]:
import shutil
shutil.make_archive("demo_outputs", "zip", OUT)
print("만들어진 파일:", sorted(p.name for p in OUT.iterdir()))
try:
    from google.colab import files
    files.download("demo_outputs.zip")
except ImportError:
    print("Colab이 아닙니다. demo_outputs.zip 을 직접 보관하세요.")

## 정리 — 내 포트폴리오 주제 고르기

| 사례 | 포트폴리오의 핵심 문장 | 직접 해 볼 확장 |
|---|---|---|
| 박스 카운팅 | "붙은 박스를 분할로 갈라 개수 오차를 ○개로 줄였다" | 직접 찍은 택배 사진 30장 추가 학습 |
| 운동·낙상 | "단순 규칙의 오탐을 시간 조건으로 줄였다" | 스쿼트(엉덩이–무릎–발목 각도)로 바꾸기 |
| 주차·대기열 | "카메라 시점에 맞는 모델과 영역 설계로 빈자리를 셌다" | 학교·학원 주차장 사진으로 영역 직접 찍기 |